# 4. Dictionaries


## 25: Be Cautious when Relying on Dictionary Insertion Ordering

### Notatka: **kwargs

- W sygnaturze funkcji zapis `**kwargs` zbiera wszystkie dodatkowe argumenty nazwane i umieszcza je w słowniku (dict) o nazwie `kwargs`.
- Do elementów można się odwoływać jak do zwykłego słownika: `kwargs['klucz']` lub iterując: `for k, v in kwargs.items(): ...`.
- Przykład:

```python
def my_func(**kwargs):
    for key, value in kwargs.items():
        print(f"{key} = {value}")

my_func(goose='gosling', kangaroo='joey')
```

- Operator `**` przy wywołaniu funkcji rozpakowuje słownik na argumenty nazwane: `f(**{'x':1, 'y':2})`.
- `kwargs` to zwykły dict; od Pythona 3.7 zachowuje kolejność wstawiania.
- Zastosowania: przekazywanie opcjonalnych parametrów, przekazywanie dalej (forwarding), elastyczne API.

### Notatki: dict vs SortedDict

- **dict (wbudowany)**
  - Przechowuje pary klucz→wartość; od Pythona 3.7 zachowuje kolejność wstawiania (insertion order).
  - Operacje: odczyt/wstawienie/usunięcie ~ O(1) średnio.
  - Metody: keys(), values(), items(), popitem(), get(), update() itp.
  - Zastosowanie: szybki dostęp po kluczu; gdy ważna jest kolejność wstawiania.

- **SortedDict (posortowany)**
  - Ogólne pojęcie: mapowanie, które iteruje w kolejności posortowanej według klucza.
  - Implementacje różne (własne klasy sortujące klucze, sortedcontainers.SortedDict, bisect+listy): różne złożoności.
  - Typowe złożoności: wstawianie/odczyt O(log n) dla struktur drzewiastych; iteracja w porządku posortowanym O(n).
  - Zastosowanie: gdy potrzebna jest zawsze posortowana iteracja lub zapytania zakresowe.

- **Główne różnice**
  - Kolejność: dict → insertion order; SortedDict → porządek posortowany według klucza.
  - Wydajność: dict jest szybszy dla operacji kluczowych; SortedDict ma większy koszt (implementacja zależna).
  - API/typ: SortedDict bywa Mappingiem, niekoniecznie instancją dict — unikaj sprawdzania isinstance(obj, dict), używaj duck-typing lub Mapping.
  - Pamięć: implementacje posortowane zwykle mają większy narzut pamięci.

- **Pułapki i wskazówki praktyczne**
  - Nie polegaj na next(iter(mapping)) jako „zwycięzcy” bez sprawdzenia typu — dla dict to element wg insertion order, dla SortedDict to najmniejszy (posortowany).
  - Jeśli iterujesz i klucze są sortowane przy każdej iteracji, pamiętaj o koszcie O(n log n).
  - Wybierz dict dla szybkości i prostoty; wybierz SortedDict (np. z paczki sortedcontainers) gdy potrzebujesz gwarantowanej posortowanej kolejności i operacji zakresowych.

- Jeśli chcesz, mogę dodać krótki benchmark porównawczy (dla Twoich danych) lub przykład implementacji SortedDict (bisect / sortedcontainers).

In [1]:
baby_names = {
    "cat": "kitten",
    "dog": "puppy"
}

print(baby_names)

{'cat': 'kitten', 'dog': 'puppy'}


In [3]:
print(list(baby_names.keys()))
print(list(baby_names.values()))
print(list(baby_names.items()))
print(baby_names.popitem())  # last inserted item

['cat', 'dog']
['kitten', 'puppy']
[('cat', 'kitten'), ('dog', 'puppy')]
('dog', 'puppy')


In [4]:
def my_func(**kwargs):
    for key, value in kwargs.items():
        print(f'{key} = {value}')

my_func(goose='gosling', kangaroo = 'joey')

goose = gosling
kangaroo = joey


In [4]:
class MyClass:
    def __init__(self):
        self.alligator = "hatchling"
        self.elephant = "calf"
        
a = MyClass()
for key, value in a.__dict__.items():
    print(f'{key} = {value}')

alligator = hatchling
elephant = calf


In [5]:
votes = {
"otter": 1281,
"polar bear": 587,
"fox": 863,
}

In [14]:
def populate_ranks(votes, ranks):
    names = list(votes.keys())
    names.sort(key=votes.get, reverse=True)
    for i, name in enumerate(names, 1):
        ranks[name] = i

In [15]:
def get_winner(ranks):
    return next(iter(ranks))

In [16]:
ranks = {}
populate_ranks(votes, ranks)
print(ranks)
winner = get_winner(ranks)
print(winner)

{'otter': 1, 'fox': 2, 'polar bear': 3}
otter


In [17]:
from collections.abc import MutableMapping

class SortedDict(MutableMapping):
    def __init__(self):
        super().__init__()
        self.data = {}
        
    def __getitem__(self, key):
        return self.data[key]
    
    def __setitem__(self, key, value):
        self.data[key] = value
        
    def __delitem__(self, key):
        del self.data[key]
        
    def __iter__(self):
        keys = list(self.data.keys())
        keys.sort()
        for key in keys:
            yield key
            
    def __len__(self):
        return len(self.data)

In [18]:
sorted_ranks = SortedDict()
populate_ranks(votes, sorted_ranks)
print(sorted_ranks.data)
winner = get_winner(sorted_ranks)
print(winner)

{'otter': 1, 'fox': 2, 'polar bear': 3}
fox


In [19]:
def get_winner(ranks):
    for name, rank in ranks.items():
        if rank == 1:
            return name
        
winner = get_winner(sorted_ranks)
print(winner)

otter


In [20]:
def get_winner(ranks):
    if not isinstance(ranks, dict):
        raise TypeError("Must provide a dict instance")
    return next(iter(ranks))

get_winner(sorted_ranks)

TypeError: Must provide a dict instance

In [22]:
from typing import Dict, MutableMapping

def populate_ranks(votes: Dict[str, int],
                   ranks: Dict[str, int]) -> None:
    names = list(votes.keys())
    names.sort(key=votes.__getitem__, reverse=True)
    for i, name in enumerate(names, 1):
        ranks[name] = i

def get_winner(ranks: Dict[str, int]) -> str:
    return next(iter(ranks))

class SortedDict(MutableMapping[str, int]):
    def __init__(self):
        self._data: Dict[str, int] = {}

    def __getitem__(self, key: str) -> int:
        return self._data[key]

    def __setitem__(self, key: str, value: int) -> None:
        self._data[key] = value

    def __delitem__(self, key: str) -> None:
        del self._data[key]

    def __iter__(self):
        for key in sorted(self._data.keys()):
            yield key

    def __len__(self) -> int:
        return len(self._data)

    def items(self):
        for key in self:
            yield (key, self._data[key])


In [23]:
votes = {
    "otter": 1281,
    "polar bear": 587,
    "fox": 863,
}


In [24]:
sorted_ranks = SortedDict()
populate_ranks(votes, sorted_ranks)
print(sorted_ranks.data)
winner = get_winner(sorted_ranks)
print(winner)

AttributeError: 'SortedDict' object has no attribute 'data'

## 26. Prefer `get` over `in` and `KeyError` to Handle Missing Dictionary Keys

In [36]:
counters = {
    "pumpernickel": 2,
    "sourdough": 1,
}

In [28]:
key = "wheat"

if key in counters:
    count = counters[key]
else:
    count = 0
    
counters[key] = count + 1

print(counters)

{'pumpernickel': 2, 'sourdough': 1, 'wheat': 2}


In [37]:
try:
    count = counters[key]
except KeyError:
    count = 0
    
counters[key] = count + 1

In [38]:
count = counters.get(key, 0)
counters[key] = count + 1

>**NOTE**
>
>If you’re maintaining dictionaries of counters like
>this, it’s worth considering the `Counter` class from
>the collections built-in module, which provides
>most of the functionality you're likely to need

In [41]:
votes = {
    "baguette": ['Bob','Alice'],
    'ciabatta': ['Coco','Deb']
}

key = 'brioche'
who = 'Elmer'

if key in votes:
    names = votes[key]
else:
    votes[key] = names = []
    
names.append(who)
print(votes)

{'baguette': ['Bob', 'Alice'], 'ciabatta': ['Coco', 'Deb'], 'brioche': ['Elmer']}


In [42]:
try:
    names = votes[key]
except KeyError:
    votes[key] = names = []
    
names.append(who)

In [43]:
names = votes.get(key)
if names is None:
    votes[key] = names = []
    
names.append(who)

In [44]:
if (names := votes.get(key)) is None:
    votes[key] = names = []
names.append(who)

In [47]:
# setdefault: jeśli 'key' istnieje w słowniku 'votes', zwraca istniejącą wartość (np. listę)
# jeśli 'key' nie istnieje, tworzy wpis votes[key] = [] (domyślna wartość) i zwraca tę nowo utworzoną listę
names = votes.setdefault(key, [])
names.append(who)
# końcowy efekt: votes[key] będzie listą zawierającą 'who'

In [48]:
data = {}
key = "foo"
value = []
data.setdefault(key, value)
print("Before:", data)
value.append("hello")
print("After: ", data)

Before: {'foo': []}
After:  {'foo': ['hello']}


## 27.  Prefer `defaultdict` over `setdefault` to Handle Missing Items in Internal State

In [50]:
visits = {
    "Mexico": {"Tulum", "Puerto Vallarta"},
    "Japan": {"Hekone"},
}

In [51]:
# Short
visits.setdefault("France", set()).add("Arles")

In [52]:
# Long
if (japan := visits.get("Japan")) is None:
    visits["Japan"] = japan = set()
    
japan.add("Kyoto")
print(visits)

{'Mexico': {'Tulum', 'Puerto Vallarta'}, 'Japan': {'Kyoto', 'Hekone'}, 'France': {'Arles'}}


In [55]:
class Visits:
    def __init__(self):
        self.data = {}
        
    def add(self, country, city):
        city_set = self.data.setdefault(country, set())
        city_set.add(city)

In [56]:
visits = Visits()
visits.add("Russia", "Yekaterinburg")
visits.add("Tanzania", "Zanzibar")
print(visits.data)

{'Russia': {'Yekaterinburg'}, 'Tanzania': {'Zanzibar'}}


In [57]:
from collections import defaultdict

class Visits:
    def __init__(self):
        self.data = defaultdict(set)
    
    def add(self, country, city):
        self.data[country].add(city)
        
visits = Visits()
visits.add("England", "Bath")
visits.add("England", "London")
print(visits.data)

defaultdict(<class 'set'>, {'England': {'London', 'Bath'}})


## 28. Know How to Construct Key-Dependent Default Values with `__missing__`

### Jak działa try/except/else/finally

- try: blok, w którym wykonuje się kod mogący rzucić wyjątek.
- except [TypWyjątku] as e: blok obsługi; wykonywany, gdy w try wystąpi wyjątek pasujący do typu. Można mieć wiele bloków except.
- else: wykonywany tylko wtedy, gdy w try nie wystąpił żaden wyjątek (przydatny do kodu zależnego od sukcesu try).
- finally: wykonywany zawsze, niezależnie od tego, czy wystąpił wyjątek — używany do sprzątania (zamknięcie plików, zwolnienie zasobów).

Kolejność wykonania: try → (opcjonalne excepty) → else (jeśli brak wyjątku) → finally (zawsze).

### Krótki przykład z wyjaśnieniem

Przykład pokazujący kolejność wykonywania bloków try/except/else/finally:

```python
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print('Błąd: dzielenie przez zero')
        return None
    else:
        print('Operacja powiodła się, zwracam wynik')
        return result
    finally:
        print('Blok finally: sprzątanie (wykonywany zawsze)')

print(divide(10, 2))  # try -> else -> finally
print(divide(5, 0))   # try -> except -> finally
```

Omówienie:
- Gdy nie wystąpi wyjątek: wykonuje się try, potem else, a na końcu finally.
- Gdy wystąpi wyjątek obsłużony przez except: wykonuje się except, potem finally (else pomijane).
- finally uruchamia się zawsze, niezależnie od zwracanej wartości lub ponownego rzucenia wyjątku — przydatne do zamykania plików i zwalniania zasobów.

In [5]:
pictures = {}
path = "profile_1234.png"

if (handle := pictures.get(path)) is None:
    try:
        handle = open(path, "a+b")
    except OSError:
        print(f'Failed to open path {path}')
        raise
    else:
        pictures[path] = handle
        
handle.seek(0)
image_date = handle.read()

Poniżej bardzo dokładne wyjaśnienie krok po kroku, co robi ten fragment kodu i jakie są konsekwencje jego użycia.

Kod (dla przypomnienia):

```python
pictures = {}
path = "profile_1234.png"

if (handle := pictures.get(path)) is None:
    try:
        handle = open(path, "a+b")
    except OSError:
        print(f'Failed to open path {path}')
        raise
    else:
        pictures[path] = handle
        
handle.seek(0)
image_date = handle.read()
```

1. Co to za obiekty na początku
   - pictures = {} — pusty słownik (dict). Będzie służył jako cache: klucz = ścieżka pliku, wartość = otwarty uchwyt pliku (file object).
   - path = "profile_1234.png" — łańcuch z nazwą/ścieżką pliku.

2. Wyrażenie (handle := pictures.get(path))
   - To jest operator przypisania wyrażenia (tzw. walrus operator, od Pythona 3.8).
   - pictures.get(path) próbuje pobrać wartość dla klucza path. Jeśli nie ma takiego klucza zwraca None.
   - handle := ... przypisuje wynik wywołania do zmiennej handle i równocześnie zwraca tę wartość do instrukcji if.
   - Zatem po tej linii handle będzie albo:
     - obiektem uchwytu pliku, jeśli wcześniej był zapisany w pictures[path], albo
     - None, gdy plik jeszcze nie był otwarty i wpis w słowniku nie istnieje.

3. if ... is None:
   - Sprawdza czy w cache (pictures) nie ma jeszcze uchwytu dla tej ścieżki.
   - Użycie is None jest poprawnym stylem do sprawdzania braku wartości.

4. Blok try:
   - Jeśli handle jest None, próbujemy otworzyć plik:
     - open(path, "a+b")
     - try/except przechwytuje OSError (np. brak uprawnień, nie istniejący folder, itp.).
   - Tryb "a+b":
     - "a" = append: otwórz do dopisywania; jeśli plik nie istnieje, utwórz go.
     - "b" = binary: plik otwierany w trybie binarnym (zwracane bajty, nie str).
     - W trybie "a" wskaźnik pliku przy otwarciu jest ustawiony na koniec pliku dla operacji pisania. Dlatego przed czytaniem trzeba zrobić seek(0) — co kod robi później.

5. except OSError:
   - Jeśli wystąpi błąd przy otwieraniu pliku, kod wypisuje komunikat i ponownie rzuca wyjątek (raise). Dzięki temu błąd nie jest „zjadany” — program/wywołujący dowie się o niepowodzeniu.

6. else:
   - Else w try/except wykonuje się tylko, gdy w try nie wystąpił wyjątek.
   - pictures[path] = handle — zapamiętujemy nowo otwarty uchwyt w słowniku, aby przy kolejnych wywołaniach reużyć ten sam uchwyt (cache).

7. Po if (poza blokiem):
   - handle.seek(0) — ustaw wskaźnik pliku na początek (potrzebne, bo przy otwarciu w "a+b" wskaźnik mógł być na końcu; a także jeśli uchwyt pochodzi z cache, jego pozycja może być gdziekolwiek).
   - image_date = handle.read() — odczytaj całe ciało pliku jako bajty i przypisz do zmiennej image_date.
     - Uwaga: najprawdopodobniej to literówka — autor chciał nazwać zmienną image_data. Literówka nie zniszczy działania, ale warto poprawić nazwę.

8. Istotne uwagi o zachowaniu i potencjalnych problemach
   - Przechowywanie uchwytów plików w słowniku:
     - Działa i przyspiesza ponowne użycie (brak ponownego open), ale wymaga zarządzania zamykaniem plików. Jeśli nigdy ich nie zamkniesz, system może wyczerpać limity uchwytów.
   - Brak zamknięcia pliku:
     - Kod nie zamyka pliku. Rozważ dodanie mechanizmu zamykającego (np. explicite close przy zakończeniu programu albo mechanizm "cleanup").
   - Wyścigi (race conditions) w wielowątkowym środowisku:
     - Jeśli ten kod może być wywoływany równolegle (różne wątki/procesy), dwie instancje mogą jednocześnie przeczytać pictures.get(path) == None i obie otworzyć plik — może powstać więcej niż jeden uchwyt. W takiej sytuacji trzeba synchronizować dostęp (Lock) lub użyć bezpiecznej fabryki, która tworzy pojedynczy wpis atomowo.
   - Walrus operator tu upraszcza kod — alternatywnie można zrobić handle = pictures.get(path); if handle is None: ...
   - Czy użyć setdefault / defaultdict / __missing__?
     - pictures.setdefault(path, open(path, "a+b")) — NIEBEZPIECZNE jeśli chcesz obsłużyć wyjątki przy open, bo open(path, ...) zostanie wywołane zanim setdefault sprawdzi istnienie klucza; więc open wykona się zawsze, nawet gdy element jest już w słowniku.
     - defaultdict(factory) — domyślna fabryka jest wywoływana bez argumentów, więc jeśli fabryka wymaga klucza (np. open_picture(path)), to tak nie zadziała. Trzeba by użyć mechanizmu, który ma dostęp do klucza (np. subclasses dict z __missing__).
     - Subclass dict z metodą __missing__(self, key) jest eleganckim rozwiązaniem: gdy klucz nie istnieje, __missing__ może otworzyć plik z użyciem klucza, zapisać do self[key] i zwrócić uchwyt. To rozwiązanie łatwo obsługuje klucz zależny od wartości i jest bardziej czytelne.
   - Race condition przy defaultdict pokazanym w notatce: w notatce użyto defaultdict(open_picture) gdzie open_picture(profile_path) oczekuje argumentu — to nie zadziała, bo default factory nie otrzymuje klucza. (To literówka / błąd koncepcyjny w notatkach.)

9. Bezpieczeństwo i poprawność I/O
   - Otwieranie pliku w "a+b" jest ok gdy chcesz tworzyć plik gdy nie istnieje i dopisywać. Jeśli plik ma być tylko do odczytu, użyj "rb".
   - Po read() otrzymujesz bajty; jeśli potrzebujesz str, trzeba je zdekodować.
   - Jeśli chcesz tylko jednorazowo odczytać i nie cachować uchwytu, lepiej użyć:

```python
with open(path, "rb") as f:
    data = f.read()
```

   - wtedy plik automatycznie się zamknie.

10. Proponowana mała poprawka (naprawa literówki i komentarz o zamykaniu):

```python
pictures = {}
path = "profile_1234.png"

# reuse existing handle if present
if (handle := pictures.get(path)) is None:
    try:
        handle = open(path, "a+b")
    except OSError:
        print(f'Failed to open path {path}')
        raise
    else:
        pictures[path] = handle

handle.seek(0)
image_data = handle.read()  # poprawiona nazwa zmiennej

# Później, przy zamykaniu programu:
for h in pictures.values():
    try:
        h.close()
    except Exception:
        pass
```

11. Krótkie podsumowanie:
   - pictures.get(path) sprawdza cache; walrus operator przypisuje wynik do handle.
   - Jeśli brak uchwytu, spróbuj otworzyć plik w try/except; przy sukcesie zapisz uchwyt w słowniku.
   - Następnie resetuj wskaźnik pliku (seek(0)) i czytaj zawartość.
   - Pamiętaj o zamykaniu uchwytów, synchronizacji w środowisku wielowątkowym i o literówce image_date → image_data.

Jeśli chcesz, mogę teraz: pokazać wersję z klasą Pictures używającą __missing__, albo dodać zamykanie/cleanup do notebooka.

In [6]:
try:
    handle = pictures.setdefault(path, open(path, "a+b"))
except OSError:
    print(f'Failed to open path {path}')
    raise
else:
    handle.seek(0)
    image_data = handle.read()

In [7]:
from collections import defaultdict

def open_picture(profile_path):
    try:
        return open(profile_path, "a+b")
    except OSError:
        print(f'Failed to open path {profile_path}')
        raise
    
pictures = defaultdict(open_picture)
handle = pictures[path]
handle.seek(0)
image_data = handle.read()

TypeError: open_picture() missing 1 required positional argument: 'profile_path'

Szczegółowe wyjaśnienie krok po kroku (dla początkującego):

1. Import:
   - from collections import defaultdict — importujemy specjalny rodzaj słownika, który może automatycznie tworzyć brakujące wartości za pomocą tzw. "fabryki" (default factory). Zwróć uwagę: fabryka jest wywoływana bez argumentów wtedy, gdy odwołujesz się do nieistniejącego klucza.

2. Funkcja open_picture(profile_path):
   - Próbuje otworzyć plik w trybie "a+b" (append + binary): jeśli plik nie istnieje, zostanie utworzony; przy otwarciu wskaźnik może być ustawiony na końcu pliku.
   - Jeśli open wyrzuci OSError (np. brak uprawnień lub ścieżka nie istnieje), funkcja wypisze komunikat i ponownie rzuci wyjątek — dzięki temu wywołujący wie o błędzie.

3. pictures = defaultdict(open_picture)
   - To deklaruje słownik, który użyje funkcji open_picture jako fabryki dla brakujących kluczy.
   - WAŻNE: fabryka (default factory) jest wywoływana BEZ ARGUMENTÓW. Ale open_picture wymaga jednego argumentu (profile_path). To powoduje problem: gdy wykonamy pictures[path], Python spróbuje wywołać open_picture() bez argumentów i dostaniemy TypeError (brak wymaganej wartości). Innymi słowy — ta linia jest błędna koncepcyjnie.

4. handle = pictures[path]
   - Gdyby fabryka była poprawna, to ta linia zwróciłaby uchwyt do pliku (file object). W tym kodzie jednak wywołanie prowadzi do wspomnianego TypeError, więc kod się zepsuje jeszcze przed seek/read.

5. handle.seek(0)
   - Jeśli mamy poprawny uchwyt, seek(0) ustawia wskaźnik pliku na początek. To potrzebne, bo otwarcie w trybie append może ustawić wskaźnik na końcu, więc przed czytaniem trzeba cofnąć go na początek.

6. image_data = handle.read()
   - Czyta cały plik i przypisuje zawartość (bajty) do zmiennej image_data.

Typowe pułapki i sugestie naprawcze:
- Nie używaj defaultdict(open_picture) jeśli open_picture potrzebuje klucza — default factory nie otrzymuje klucza.
- Proste i bezpieczne rozwiązania:
  - Użyć zwykłego dict i wcześniejszego wzorca: najpierw sprawdzić pictures.get(path) i jeśli None, otworzyć plik w try/except i zapisać do słownika.
  - Zaimplementować klasę dziedziczącą po dict z metodą __missing__(self, key), która przy braku klucza wywoła open_picture(key), zapisze wynik i zwróci uchwyt. To jest eleganckie i poprawne, bo __missing__ otrzymuje klucz.
- Ostrożność przy setdefault: pictures.setdefault(path, open(path, "a+b")) nie jest bezpieczne jeśli open może rzucić wyjątek ani jeśli chcesz uniknąć wywołania open gdy klucz istnieje — open(...) zostanie wykonane zawsze zanim setdefault sprawdzi istnienie klucza.
- Zarządzanie zasobami: pamiętaj o zamykaniu uchwytów do plików (close) gdy nie są już potrzebne, aby nie wyczerpać limitów systemowych.
- Wątki: w środowisku wielowątkowym ten wzorzec wymaga synchronizacji (Lock), inaczej dwie wątki mogą jednocześnie próbować otworzyć ten sam plik.

Krótko: ten fragment chce użyć automatycznej fabryki defaultdict do otwierania plików, ale jest błąd — fabryka nie dostaje klucza. Poprawne opcje to: użyć dict z ręcznym sprawdzeniem, albo użyć dict-subclass z __missing__, albo innego mechanizmu, który otrzymuje klucz przed wywołaniem open_picture.

In [8]:
class Pictures(dict):
    def __missing__(self, key):
        value = open_picture(key)
        self[key] = value
        return value
    
pictures = Pictures()
handle = pictures[path]
handle.seek(0)
image_data = handle.read()

## Wyjaśnienie: `defaultdict` vs `setdefault`

- `dict.setdefault(key, default)`
  - Działa tak: jeśli `key` nie istnieje w słowniku, ustawia `d[key] = default` i zwraca tę wartość; jeśli klucz jest, zwraca istniejącą wartość.
  - Pułapka: wartość `default` jest obliczana PRZED wywołaniem metody. Przykład problemu:

```python
# expensive() wykona się zawsze, nawet jeśli 'k' już istnieje
d.setdefault('k', expensive())
```

  - Użyteczne gdy masz już przygotowaną, tanią wartość do wstawienia. Uważaj na współdzielone obiekty mutowalne przekazywane jako `default`.

- `collections.defaultdict(factory)`
  - Tworzy słownik, który automatycznie tworzy brakującą wartość przez wywołanie `factory()` TYLKO gdy odczytujesz `d[key]` i klucz nie istnieje.
  - `factory` jest wywoływana bez argumentów, więc `defaultdict(list)` tworzy nową listę dla każdego brakującego klucza.
  - Zaletą jest, że factory nie jest wywoływana, dopóki klucz faktycznie nie zostanie odczytany.
  - Ograniczenie: fabryka nie otrzymuje klucza — nie nadaje się, gdy wartość domyślna zależy od klucza.

- Kiedy co wybrać (praktycznie):
  - Do automatycznego tworzenia prostych kontenerów użyj `defaultdict` (np. `defaultdict(int)`, `defaultdict(list)`, `defaultdict(set)`).
  - Gdy chcesz wstawić konkretną, już obliczoną wartość tylko jeśli klucz brak — `setdefault` może być wygodne, ale NIE używaj go gdy obliczenie default jest kosztowne lub ma skutki uboczne.
  - Gdy wartość domyślna zależy od klucza — zaimplementuj `__missing__(self, key)` w podklasie `dict` lub jawnie sprawdzaj `if key in d: ... else: d[key] = compute(key)`.

- Wielowątkowość i zasoby:
  - Ani `defaultdict`, ani `setdefault` nie zapobiegną race condition — przy równoległym dostępie może powstać podwójne tworzenie/ustawianie; użyj Lock w środowisku wielowątkowym.
  - Nie używaj `setdefault` z funkcjami, które otwierają pliki lub mają efekty uboczne — `default` zostanie obliczony zawsze.

- Krótkie przykłady:

```python
from collections import defaultdict
# defaultdict — lista tworzona tylko gdy odczytujemy brakujący klucz
g = defaultdict(list)
g['x'].append(1)

# setdefault — domyślna wartość utworzona wcześniej (wykonana przy wywołaniu)
d = {}
d.setdefault('p', []).append(2)
```

Podsumowanie: używaj `defaultdict` dla prostych, bezargowych wartości tworzonych na żądanie; używaj `setdefault` tylko gdy masz gotowy, tani do wstawienia obiekt. Dla wartości zależnych od klucza wybierz `__missing__` lub jawne sprawdzenie i przypisanie.

In [9]:
pictures

{'profile_1234.png': <_io.BufferedRandom name='profile_1234.png'>}

## 29. Compose Classes Instead of Deeply Nesting Dictionaries, Lists, and Tuples

In [10]:
class SimpleGradebook:
    def __init__(self):
        self._grades = {}
        
    def add_student(self, name):
        self._grades[name] = []
        
    def report_grade(self, name, score):
        self._grades[name].append(score)
        
    def average_grade(self, name):
        grades = self._grades[name]
        return sum(grades) / len(grades)

In [11]:
book = SimpleGradebook()
book.add_student("Isaac Newton")
book.report_grade("Isaac Newton", 90)
book.report_grade("Isaac Newton", 95)
book.report_grade("Isaac Newton", 85)
print(book.average_grade("Isaac Newton"))

90.0


In [14]:
from collections import defaultdict

class BySubjectGradebook:
    def __init__(self):
        self._grades = {}
        
    def add_student(self, name):
        self._grades[name] = defaultdict(list)
        
    def report_grade(self, name, subject, grade):
        by_subject = self._grades[name]
        grade_list = by_subject[subject]
        grade_list.append(grade)
        
    def average_grade(self, name):
        by_subject = self._grades[name]
        
        total, count = 0, 0
        for grades in by_subject.values():
            total += sum(grades)
            count += len(grades)
            
        return total / count

In [15]:
book = BySubjectGradebook()
book.add_student("Albert Einstein")
book.report_grade("Albert Einstein", "Math", 75)
book.report_grade("Albert Einstein", "Math", 65)
book.report_grade("Albert Einstein", "Gym", 90)
book.report_grade("Albert Einstein", "Gym", 95)
print(book.average_grade("Albert Einstein"))


81.25


In [17]:
class WeightedGradebook:
    def __init__(self):
        self._grades = {}
        
    def add_student(self, name):
        self._grades[name] = defaultdict(list)
        
    def report_grade(self, name, subject, score, weight):
        by_subject = self._grades[name]
        grade_list = by_subject[subject]
        grade_list.append((score, weight))
        
    def average_grade(self, name):
        by_subject = self._grades[name]
        
        score_sum, score_count = 0, 0
        for scores in by_subject.values():
            subject_avg, total_weight = 0, 0
            for score, weight in scores:
                subject_avg += score * weight
                total_weight += weight
            
            score_sum += subject_avg / total_weight
            score_count += 1
        
        return score_sum / score_count

In [18]:
book = WeightedGradebook()
book.add_student("Albert Einstein")
book.report_grade("Albert Einstein", "Math", 75, 0.05)
book.report_grade("Albert Einstein", "Math", 65, 0.15)
book.report_grade("Albert Einstein", "Math", 70, 0.80)
book.report_grade("Albert Einstein", "Gym", 100, 0.40)
book.report_grade("Albert Einstein", "Gym", 85, 0.60)
print(book.average_grade("Albert Einstein"))


80.25


### Refactoring to Classes

In [22]:
grades = []
grades.append((95, 0.45))
grades.append((85, 0.55))
total = sum(score * weight for score, weight in grades)
total_weight = sum(weight for _, weight in grades)
average_grade = total / total_weight
average_grade

89.5

In [23]:
grades = []
grades.append((95, 0.45, "Great job"))
grades.append((85, 0.55, "Better next time"))
total = sum(score * weight for score, weight, _ in grades)
total_weight = sum(weight for _, weight, _ in grades)
average_grade = total / total_weight
average_grade

89.5

### Co to jest `dataclass` i po co go używać?

`dataclass` to specjalny dekorator w Pythonie (od wersji 3.7), który automatycznie generuje podstawowe metody dla klasy, takie jak `__init__`, `__repr__`, `__eq__` i inne. Dzięki temu nie musisz ręcznie pisać powtarzalnego kodu do przechowywania danych.

#### Po co używać `dataclass`?

- **Prostota**: Pozwala szybko zdefiniować klasę do przechowywania danych (np. rekord, konfiguracja, wynik pomiaru) bez pisania żmudnych konstruktorów.
- **Czytelność**: Kod jest krótszy i bardziej przejrzysty — od razu widać, jakie pola ma obiekt.
- **Automatyczne metody**: Python sam tworzy metody porównania (`__eq__`), wyświetlania (`__repr__`), kopiowania itp.
- **Typowanie**: Możesz (ale nie musisz) dodać typy pól, co ułatwia wykrywanie błędów i współpracę z narzędziami typu linters.

#### Przykład

```python
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int

p = Person("Ala", 30)
print(p)  # Person(name='Ala', age=30)
```

**Podsumowanie:**  
Używaj `@dataclass`, gdy chcesz szybko i wygodnie stworzyć klasę do przechowywania danych, bez pisania zbędnego kodu.

In [24]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Grade:
    score:int
    weight: float

In [25]:
class Subject:
    def __init__(self):
        self._grades = []
        
    def report_grade(self, score, weight):
        self._grades.append(Grade(score, weight))
        
    def average_grade(self):
        total, total_weight = 0, 0
        for grade in self._grades:
            total += grade.score * grade.weight
            total_weight += grade.weight
        return total / total_weight

In [26]:
class Student:
    def __init__(self):
        self._subjects = defaultdict(Subject)
        
    def get_subject(self, name):
        return self._subjects[name]
    
    def average_grade(self):
        total, count = 0, 0
        for subject in self._subjects.values():
            total += subject.average_grade()
            count += 1
        return total / count

In [27]:
class Gradebook:
    def __init__(self):
        self._students = defaultdict(Student)
        
    def get_student(self, name):
        return self._students[name]

In [29]:
book = Gradebook()
albert = book.get_student("Albert Einstein")
math = albert.get_subject("Math")
math.report_grade(75, 0.05)
math.report_grade(65, 0.15)
math.report_grade(70, 0.80)
gym = albert.get_subject("Gym")
gym.report_grade(100, 0.40)
gym.report_grade(85, 0.60)
print(albert.average_grade())


80.25


# Ćwiczenie do Item 25–29: słowniki, wartości domyślne i kompozycja klas

## Opis:
Poznaj dobre praktyki pracy ze słownikami w Pythonie: kolejność wstawek, obsługa brakujących kluczy, różnice między `defaultdict` i `setdefault`, `__missing__`, oraz kiedy zamiast zagnieżdżonych struktur użyć kompozycji klas.

## Zadania:
1) Kolejność wstawek (Item 25)
   - Zaimplementuj prosty rejestr zdarzeń `EventLog`, który:
     * dodaje wpisy w kolejności nadejścia,
     * potrafi zwrócić n ostatnich (FIFO) bez sortowania.
   - Pokaż, że mutacja istniejącej wartości nie zmienia kolejności kluczy.
   - (Opcjonalnie) Porównaj z `OrderedDict` (czy coś się dziś różni?).

2) Obsługa braków kluczy: `get` vs `in`/`KeyError` (Item 26)
   - Napisz funkcję `read_config(dct, key, default="MISSING")`, która:
     * używa `dict.get` do zwrotu wartości lub domyślnej,
     * loguje, gdy klucz nie istnieje (bez łapania wyjątku).
   - Dodaj wersję opartą o `try/except KeyError` i porównaj czytelność.

3) `defaultdict` vs `setdefault` (Item 27)
   - Zaimplementuj zliczanie słów w tekście:
     * Wersja A: `collections.defaultdict(int)`,
     * Wersja B: zwykły `dict` + `setdefault`.
   - Zmierz prosty czas (np. `timeit`) i pokaż różnice w kodzie/ergonomii.

4) `__missing__` dla wartości zależnych od klucza (Item 28)
   - Stwórz klasę `AutoDict(dict)`:
     * dla nieistniejącego klucza zwraca komunikat `f"<missing:{key}>"` (nie dodaje go),
     * dodaj wariant `ComputeDict(dict)` wyliczający wartość np. `len(key)` w `__missing__` i zapisujący ją.

5) Kompozycja klas zamiast głębokich struktur (Item 29)
   - Masz dane:
     ```python
     order = {
       "id": "A-100",
       "customer": {"id": 7, "name": "Alice"},
       "items": [{"sku": "X1", "qty": 2, "price": 10.0}, {"sku": "Z9", "qty": 1, "price": 99.0}]
     }
     ```
   - Zdefiniuj `@dataclass`-y: `Customer`, `Item`, `Order` z metodą `total()`.
   - Zaimplementuj konstruktor klasowy `Order.from_dict(order_dict)` (Item 52/kompozycja).
   - Pokaż przewagi: typowanie, enkapsulacja logiki i testowalność.

## Cel:
- Świadomie korzystać z kolejności w `dict`.
- Używać `get`/`try-except` adekwatnie do sytuacji.
- Znać różnice praktyczne między `defaultdict`, `setdefault` i `__missing__`.
- Zamieniać „spaghetti” zagnieżdżeń na czytelne klasy z metodami.


# Exercise for Item 25–29: Dictionaries, default values, and class composition

## Description:
Explore best practices when working with Python dictionaries: insertion order, handling missing keys, differences between `defaultdict` and `setdefault`, the `__missing__` hook, and when to replace deeply nested structures with class composition.

## Tasks:
1) Insertion order (Item 25)
   - Implement a simple event log `EventLog` that:
     * adds entries in insertion order,
     * can return the last n entries without sorting.
   - Show that mutating an existing value does not affect key order.
   - (Optional) Compare with `OrderedDict` (what’s different today?).

2) Handling missing keys: `get` vs `in`/`KeyError` (Item 26)
   - Write a function `read_config(dct, key, default="MISSING")` that:
     * uses `dict.get` to return a value or a default,
     * logs when the key is absent (without catching an exception).
   - Add a version based on `try/except KeyError` and compare readability.

3) `defaultdict` vs `setdefault` (Item 27)
   - Implement word counting:
     * Version A: `collections.defaultdict(int)`,
     * Version B: plain `dict` + `setdefault`.
   - Measure simple timing (e.g., `timeit`) and compare code clarity.

4) `__missing__` for key-dependent defaults (Item 28)
   - Create a class `AutoDict(dict)` that:
     * for a missing key returns a message `f"<missing:{key}>"` (without adding it),
     * create a variant `ComputeDict(dict)` that computes a value (e.g., `len(key)`) in `__missing__` and stores it.

5) Class composition instead of deep nesting (Item 29)
   - Given data:
     ```python
     order = {
       "id": "A-100",
       "customer": {"id": 7, "name": "Alice"},
       "items": [
           {"sku": "X1", "qty": 2, "price": 10.0},
           {"sku": "Z9", "qty": 1, "price": 99.0}
       ]
     }
     ```
   - Define `@dataclass`es: `Customer`, `Item`, `Order` with a method `total()`.
   - Implement a classmethod `Order.from_dict(order_dict)`.
   - Show the advantages: type safety, encapsulated logic, and testability.

## Goal:
- Use insertion order in `dict` consciously.
- Handle missing keys with `get`/`try-except` appropriately.
- Understand practical differences between `defaultdict`, `setdefault`, and `__missing__`.
- Replace “spaghetti” nested dicts/lists with clean class composition.


In [56]:
from collections import OrderedDict, deque
class EventLog:
    def __init__(self):
        self._logs = {}
        self._seqno = 1

    def addlog(self, eventlog):
        self._logs[self._seqno] = eventlog
        self._seqno += 1

    def items(self):
        return list(self._logs.items())
    
    def show_last_n(self, n = 1):
        if n <= 0:
            return []
        return list(deque(self._logs.items(), maxlen=n))
    
    def edit_event(self, seqno, new_event):
        self._logs[seqno] = new_event

In [57]:
events = EventLog()

In [58]:
events.addlog('jk')
events.addlog('jp')
events.addlog('jd')
events.addlog('jx')


In [59]:
events.show_last_n(3)

[(2, 'jp'), (3, 'jd'), (4, 'jx')]

In [45]:
events.items()

[(1, 'jk'), (2, 'jp'), (3, 'jd'), (4, 'jx')]

In [46]:
events.edit_event(2, 'japitole')

In [47]:
events.items()

[(1, 'jk'), (2, 'japitole'), (3, 'jd'), (4, 'jx')]

In [60]:
class EventLogOrdered(OrderedDict):
    def __init__(self):
        super().__init__()
        self._seqno = 1

    def addlog(self, eventlog):
        self._seqno += 1
        self[self._seqno] = eventlog

    def items(self):
        return list(super().items())
    
    def show_last_n(self, n = 1):
        if n <= 0:
            return []
        return list(deque(self.items(), maxlen=n))
    
    def edit_event(self, seqno, new_event):
        if seqno not in self:
            raise KeyError(f'No event with seqno {seqno}')
        self[seqno] = new_event

In [62]:
events_ordered = EventLogOrdered()
events_ordered.addlog('jk')
events_ordered.addlog('jp')
events_ordered.addlog('jd')
events_ordered.addlog('jx')

events_ordered.show_last_n(3)

events_ordered.items()

events.edit_event(2, 'japitole')

events.items()

[(1, 'jk'), (2, 'japitole'), (3, 'jd'), (4, 'jx')]

2) Obsługa braków kluczy: `get` vs `in`/`KeyError` (Item 26)
   - Napisz funkcję `read_config(dct, key, default="MISSING")`, która:
     * używa `dict.get` do zwrotu wartości lub domyślnej,
     * loguje, gdy klucz nie istnieje (bez łapania wyjątku).
   - Dodaj wersję opartą o `try/except KeyError` i porównaj czytelność.

In [73]:
def read_config_v1(dct: dict, key, default="MISSING"):
    return dct.get(key, default)

In [85]:
def read_config_v2(dct: dict, key, default="MISSING"):
    try:
        return dct[key]
    except KeyError:
        print(f"{key} is {default}")
        return default

In [86]:
slow = {'jeden': 1,
        'dwa': 2}

In [87]:
read_config_v1(slow, 'dwa')

2

In [88]:
read_config_v2(slow, 'trzy')

trzy is MISSING


'MISSING'

3) `defaultdict` vs `setdefault` (Item 27)
   - Zaimplementuj zliczanie słów w tekście:
     * Wersja A: `collections.defaultdict(int)`,
     * Wersja B: zwykły `dict` + `setdefault`.
   - Zmierz prosty czas (np. `timeit`) i pokaż różnice w kodzie/ergonomii.

### Wyjaśnienie kodu z benchmarkiem zliczania słów (`defaultdict` vs `setdefault`)

Poniżej znajduje się kod, który porównuje dwa sposoby zliczania słów w tekście w Pythonie oraz mierzy ich wydajność.

#### 1. Importy i przygotowanie

- `re` — moduł do pracy z wyrażeniami regularnymi (szukanie słów w tekście).
- `defaultdict` — specjalny słownik z `collections`, który automatycznie tworzy domyślne wartości.
- `timeit` — narzędzie do mierzenia czasu wykonania kodu.
- `statistics` — do obliczania średniej i mediany z wyników pomiarów.

#### 2. Funkcje zliczające słowa

- **count_words_defaultdict(text):**
    - Tworzy słownik `d` typu `defaultdict(int)`. Oznacza to, że jeśli próbujemy dodać do nieistniejącego klucza, automatycznie pojawi się tam liczba 0.
    - Dla każdego słowa w tekście zwiększa licznik o 1.
    - Zwraca słownik z liczbą wystąpień każdego słowa.

- **count_words_setdefault(text):**
    - Tworzy zwykły słownik `d`.
    - Dla każdego słowa sprawdza, czy już istnieje w słowniku. Jeśli nie, ustawia wartość 0 (`setdefault`).
    - Następnie zwiększa licznik o 1.
    - Zwraca słownik z liczbą wystąpień każdego słowa.

#### 3. Przykładowy tekst

- `sample_text` to wielokrotnie powtórzone zdanie, aby test był bardziej miarodajny.

#### 4. Sprawdzenie poprawności

- `assert` sprawdza, czy oba sposoby dają identyczny wynik (po zamianie na zwykły słownik).

#### 5. Funkcja do mierzenia czasu

- **bench(func, text, repeat=5, number=200):**
    - Wykonuje podaną funkcję wiele razy i mierzy czas.
    - Zwraca średni i medianę czasu z kilku powtórzeń.

#### 6. Benchmark — porównanie szybkości

- Dla obu funkcji (`defaultdict` i `setdefault`) mierzymy czas wykonania i wypisujemy wyniki.
- Wyniki pokazują, która metoda jest szybsza.

#### 7. Przykład użycia

- Na końcu pokazany jest przykład zliczania słów w krótkim zdaniu.

---

**Podsumowanie:**
- Obie metody służą do tego samego — liczenia, ile razy każde słowo występuje w tekście.
- `defaultdict(int)` jest wygodniejszy i zwykle szybszy, bo nie trzeba sprawdzać, czy klucz już istnieje.
- `setdefault` działa podobnie, ale jest mniej czytelny i może być wolniejszy.
- Benchmark pokazuje, która metoda działa szybciej na dużych danych.

In [89]:
import re
from collections import defaultdict
import timeit
import statistics

def count_words_defaultdict(text):
    """Wersja A: używa defaultdict(int)."""
    words = re.findall(r"\w+", text.lower())
    d = defaultdict(int)
    for w in words:
        d[w] += 1
    return d

def count_words_setdefault(text):
    """Wersja B: zwykły dict + setdefault."""
    words = re.findall(r"\w+", text.lower())
    d = {}
    for w in words:
        d.setdefault(w, 0)
        d[w] += 1
    return d

# przykładowy tekst (powtórzony, żeby mieć sensowny rozmiar danych)
sample_text = ("To jest prosty test. Sprawdzamy zliczanie słów, CASE i interpunkcja! ") * 200

# sanity check: wyniki muszą być identyczne (porównujemy po konwersji do zwykłego dict)
assert dict(count_words_defaultdict(sample_text)) == dict(count_words_setdefault(sample_text))

def bench(func, text, repeat=5, number=200):
    """Mierzy czas wykonania func(text).
       Zwraca (mean_time, median_time) z kilku powtórzeń."""
    times = timeit.repeat(lambda: func(text), repeat=repeat, number=number)
    return statistics.mean(times), statistics.median(times)

# uruchomienie benchmarku i wypisanie wyników
for name, func in (("defaultdict", count_words_defaultdict),
                   ("setdefault ", count_words_setdefault)):
    mean_t, med_t = bench(func, sample_text)
    print(f"{name:10s} mean={mean_t:.6f}s  median={med_t:.6f}s  (number=200)")

# krótki przykład użycia
print(count_words_defaultdict("Ala ma kota. Ala ma psa."))

defaultdict mean=0.115918s  median=0.113078s  (number=200)
setdefault  mean=0.127984s  median=0.128609s  (number=200)
defaultdict(<class 'int'>, {'ala': 2, 'ma': 2, 'kota': 1, 'psa': 1})


4) `__missing__` dla wartości zależnych od klucza (Item 28)
   - Stwórz klasę `AutoDict(dict)`:
     * dla nieistniejącego klucza zwraca komunikat `f"<missing:{key}>"` (nie dodaje go),
     * dodaj wariant `ComputeDict(dict)` wyliczający wartość np. `len(key)` w `__missing__` i zapisujący ją.

In [90]:
# cell: implementacja zadania 4 — __missing__ (AutoDict i ComputeDict)
class AutoDict(dict):
    """Dla brakującego klucza zwraca f'<missing:{key}>' — NIE dodaje wpisu do słownika."""
    def __missing__(self, key):
        return f"<missing:{key}>"

class ComputeDict(dict):
    """Dla brakującego klucza oblicza wartość (tu: len(key)), zapisuje ją i zwraca."""
    def __missing__(self, key):
        value = len(key)
        self[key] = value
        return value

# przykłady i proste asercje / sanity checks
ad = AutoDict()
# dostep przez __getitem__ - __missing__ zwraca specjalny komunikat, nie dodaje klucza
assert ad['nope'] == "<missing:nope>"
assert 'nope' not in ad  # klucz nie został zapisany

cd = ComputeDict()
# dostep przez __getitem__ - __missing__ oblicza i zapisuje wartość
assert cd['abc'] == 3
assert cd['abc'] == 3  # teraz klucz istnieje
assert 'abc' in cd

# pokazowe wydruki
print("AutoDict['x'] ->", AutoDict()['x'])
print("ComputeDict before:", dict(ComputeDict()))  # pusty
tmp = ComputeDict()
print("ComputeDict['hello'] ->", tmp['hello'], "   after:", tmp)

AutoDict['x'] -> <missing:x>
ComputeDict before: {}
ComputeDict['hello'] -> 5    after: {'hello': 5}


5) Kompozycja klas zamiast głębokich struktur (Item 29)
   - Masz dane:
     ```python
     order = {
       "id": "A-100",
       "customer": {"id": 7, "name": "Alice"},
       "items": [{"sku": "X1", "qty": 2, "price": 10.0}, {"sku": "Z9", "qty": 1, "price": 99.0}]
     }
     ```
   - Zdefiniuj `@dataclass`-y: `Customer`, `Item`, `Order` z metodą `total()`.
   - Zaimplementuj konstruktor klasowy `Order.from_dict(order_dict)` (Item 52/kompozycja).
   - Pokaż przewagi: typowanie, enkapsulacja logiki i testowalność.

In [91]:
# cell: implementacja zadania 5 — dataclasses Customer, Item, Order + Order.from_dict
from dataclasses import dataclass
from typing import List

@dataclass(frozen=True)
class Customer:
    id: int
    name: str

@dataclass(frozen=True)
class Item:
    sku: str
    qty: int
    price: float

    def total(self) -> float:
        return self.qty * self.price

@dataclass
class Order:
    id: str
    customer: Customer
    items: List[Item]

    def total(self) -> float:
        return sum(item.total() for item in self.items)

    @classmethod
    def from_dict(cls, d: dict) -> "Order":
        cust = Customer(**d["customer"])
        items = [Item(**it) for it in d.get("items", [])]
        return cls(id=d["id"], customer=cust, items=items)

# przykład użycia i proste asercje
order_dict = {
    "id": "A-100",
    "customer": {"id": 7, "name": "Alice"},
    "items": [
        {"sku": "X1", "qty": 2, "price": 10.0},
        {"sku": "Z9", "qty": 1, "price": 99.0},
    ],
}

order = Order.from_dict(order_dict)
assert abs(order.total() - 119.0) < 1e-9
print(order)
print("Total:", order.total())

Order(id='A-100', customer=Customer(id=7, name='Alice'), items=[Item(sku='X1', qty=2, price=10.0), Item(sku='Z9', qty=1, price=99.0)])
Total: 119.0


In [1]:
lista = [4,7,1,2,8,23,3]

In [2]:
lista.sort()

In [3]:
lista

[1, 2, 3, 4, 7, 8, 23]

In [4]:
lista = [4,7,1,2,8,23,3]
sorted(lista)

[1, 2, 3, 4, 7, 8, 23]

In [5]:
lista

[4, 7, 1, 2, 8, 23, 3]

### Deques and other queues

In [6]:
from collections import deque

In [26]:
dq = deque(range(10), maxlen=10)
dq

deque([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], maxlen=10)

In [27]:
dq.rotate(3)
dq

deque([7, 8, 9, 0, 1, 2, 3, 4, 5, 6], maxlen=10)

In [28]:
dq.rotate(-4)
dq

deque([1, 2, 3, 4, 5, 6, 7, 8, 9, 0], maxlen=10)

In [29]:
dq.appendleft(-1)
dq

deque([-1, 1, 2, 3, 4, 5, 6, 7, 8, 9], maxlen=10)

In [30]:
dq.extend([11,22,33])
dq

deque([3, 4, 5, 6, 7, 8, 9, 11, 22, 33], maxlen=10)

In [31]:
dq.extendleft([10,20,30,40])
dq

deque([40, 30, 20, 10, 3, 4, 5, 6, 7, 8], maxlen=10)

### Dunder-method map: list | array.array | collections.deque

Poniżej skrócona, linia-w-linię mapa najważniejszych metod magicznych (dunder). ✅ = pełne wsparcie, ▶ = częściowe / ograniczone, — = brak.

Method | list | array.array | collections.deque
---|---:|---:|---:
__len__ | ✅ | ✅ | ✅
__getitem__ | ✅ (index i slice) | ✅ (index i slice) | ✅ (index; slicing limited / no slice)
__setitem__ | ✅ (index i slice) | ✅ (index i slice) | ▶ (index assignment supported; slice assignment not)
__delitem__ | ✅ (index i slice) | ✅ (index i slice) | — (brak usuwania przez indeks)
__iter__ | ✅ | ✅ | ✅
__reversed__ | ✅ | ✅ | ✅
__contains__ | ✅ | ✅ | ✅
__add__ / __radd__ (concat) | ✅ (list + list) | ✅ (same-type arrays) | —
__iadd__ | ✅ | ✅ | ✅ (extend / in-place extend)
__mul__ / __rmul__ | ✅ | ✅ | —
__imul__ | ✅ | ✅ | —
__repr__ / __str__ | ✅ | ✅ | ✅
__eq__ / __ne__ | ✅ | ✅ | ✅ (equality compares elements)
Ordering (__lt__, __le__, __gt__, __ge__) | ✅ (lexicographic) | ✅ (lexicographic) | — (generally not supported)
__hash__ | — (unhashable) | — (unhashable) | — (unhashable)
Buffer protocol / memory view | — | ✅ (supports buffer / tobytes) | —

Uwagi krótkie:
- "▶ częściowe" oznacza ograniczenia (np. deque indeksowanie O(n), brak slice-assign).
- array.array wymienia specyficzne metody (tobytes, frombytes, buffer_info) — to są metody, nie dundery.
- Implementacja może się różnić w zależności od wersji Pythona; tabela zawiera typowe, publiczne dundery na CPython.

---

### Common (non-dunder) methods: list | array.array | collections.deque

Poniżej linia-w-linię najczęściej używanych metod (nie-magic). ✅ = dostępne, ▶ = ograniczenia, — = brak.

Method | list | array.array | collections.deque
---|---:|---:|---:
append(x) | ✅ | ✅ | ✅
extend(iterable) | ✅ | ✅ | ✅
insert(i, x) | ✅ | ✅ | —
remove(value) | ✅ | ✅ | ✅ (removes first matching)
pop([i]) | ✅ (end or index) | ✅ (index only) | ✅ (pop() / popleft())
clear() | ✅ | ✅ | ✅
index(value, start=..., end=...) | ✅ | ✅ | ✅ (O(n))
count(value) | ✅ | ✅ | ✅
sort(key=None, reverse=False) | ✅ | — | —
reverse() | ✅ | — | —
copy() | ✅ | ▶ (use tolist()/array(...)) | ✅
buffer_info() | — | ✅ | —
byteswap() | — | ✅ | —
frombytes(bytes) / tobytes() | — | ✅ | —
fromlist(list) / tolist() | — | ✅ | —
appendleft(x) | — | — | ✅
popleft() | — | — | ✅
extendleft(iterable) | — | — | ✅
rotate(n=1) | — | — | ✅
maxlen (attribute) | — | — | ✅

Uwagi krótkie:
- deque indeksowanie i operacje oparte na indeksie są O(n); preferuj popleft/appendleft/append/pop/rotate.
- array.array przechowuje homogeniczne typy — nie wszystkie listowe operacje są sensowne (np. sort działa, ale typy muszą być porównywalne).
- copy() dla array można wykonać przez array(typecode, original) lub użyć .tolist()+array(...).

Jeśli chcesz, wstawię przykład kodu demonstrujący różnice lub zapiszę tę tabelę jako CSV w notebooku.

In [32]:
l = [28, 14, '28', 5, '9','1', 0, 6, '23', 19]

In [33]:
sorted(l)

TypeError: '<' not supported between instances of 'str' and 'int'

In [34]:
sorted(l, key=int)

[0, '1', 5, 6, '9', 14, 19, '23', 28, '28']

In [35]:
sorted(l, key=str)

[0, '1', 14, 19, '23', 28, '28', 5, 6, '9']

## Fluent Python

### dict Comprehensions

In [37]:
dial_codes = [
    (880, 'Bangladesh'),
    (55, 'Brazil'),
    (86, 'China'),
    (91, 'India'),
    (61, 'Indonesia'),
    (81, 'Japan'),
    (234, 'Nigeria'),
    (92, 'Pakistan'),
    (7, 'Russia'),
    (1, 'United States'),
]

In [39]:
country_dial = {country: code for code, country in dial_codes}
country_dial

{'Bangladesh': 880,
 'Brazil': 55,
 'China': 86,
 'India': 91,
 'Indonesia': 61,
 'Japan': 81,
 'Nigeria': 234,
 'Pakistan': 92,
 'Russia': 7,
 'United States': 1}

In [41]:
{code: country.upper()
 for country, code in sorted(country_dial.items())
 if code < 70}

{55: 'BRAZIL', 61: 'INDONESIA', 7: 'RUSSIA', 1: 'UNITED STATES'}

### Unpacking Mappings

In [45]:
def dump(**kwargs):
    return kwargs

dump(**{'x': 1}, y=2, **{'z': 3})

{'x': 1, 'y': 2, 'z': 3}

In [43]:
{'a': 0, **{'x': 1}, 'y': 2, **{'z': 3, 'x': 4}}

{'a': 0, 'x': 4, 'y': 2, 'z': 3}

### Merging Mapping with |

In [48]:
d1 = {'a': 1, 'b': 3}
d2 = {'a': 2, 'b': 4, 'c': 6}
d2 | d1, d1 | d2

({'a': 1, 'b': 3, 'c': 6}, {'a': 2, 'b': 4, 'c': 6})

### Pattern Matching with Mappings

In [57]:
def get_creators(record: dict) -> list:
    match record:
        case {'type': 'book', 'api': 2, 'authors': [*names]}:
            return names
        case {'type': 'book', 'api': 1, 'author': name}:
            return [name]
        case {'type': 'book'}:
            raise ValueError(f"Invalid 'book' record: {record!r}")
        case {'type': 'movie', 'director': name}:
            return [name]
        case _:
            raise ValueError(f'Invalid record: {record!r}')

In [51]:
b1 = dict(api = 1, author='Douglas Hofstadter', type='book', title='Godel, Escher, Bach')
b1

{'api': 1,
 'author': 'Douglas Hofstadter',
 'type': 'book',
 'title': 'Godel, Escher, Bach'}

In [52]:
get_creators(b1)

['Douglas Hofstadter']

In [53]:
from collections import OrderedDict
b2 = OrderedDict(api = 2, type = 'book', title = 'Python in a Nutshell',
                 authors = 'Martelli Ravenscroft Holden'.split())
b2

OrderedDict([('api', 2),
             ('type', 'book'),
             ('title', 'Python in a Nutshell'),
             ('authors', ['Martelli', 'Ravenscroft', 'Holden'])])

In [54]:
get_creators(b2)

['Martelli', 'Ravenscroft', 'Holden']

In [58]:
get_creators({'type': 'book', 'pages': 770})

ValueError: Invalid 'book' record: {'type': 'book', 'pages': 770}

In [59]:
get_creators('Spam, spam, spam')

ValueError: Invalid record: 'Spam, spam, spam'

In [60]:
food = dict(category = 'ice cream', flavor ='vanilla', cost = 190)
match food:
    case {'category': 'ice cream', **details}:
        print(f'Ice cream details: {details}')

Ice cream details: {'flavor': 'vanilla', 'cost': 190}
